# SENTINEL — Train on Colab

Goal: produce the four artifacts the predict service loads at startup:

```
models/model.pkl       # CalibratedClassifierCV wrapping the stacked ensemble
models/mapie.pkl       # MAPIE conformal classifier for prediction intervals
models/schema.json     # feature order, categorical/numeric split, target name
models/metrics.json    # Brier / AUC-ROC / AUC-PR / calibration curve / SHAP top-15
models/fairness.json   # SPD, Disparate Impact, FPR/FNR parity per Race & Gender
```

Runtime: **Free Colab CPU** — no GPU needed. End-to-end < 2 minutes. Use a GPU runtime only if you scale up to a deep model later.

**Workflow:**
1. Clone the repo (replace the URL with your fork).
2. Install ML deps.
3. Generate the dataset.
4. Train.
5. Sanity-check the artifacts.
6. Download `models/` and commit them to the repo.

When the predict service starts, it will pick up `models/model.pkl` and stop using the heuristic.

In [ ]:
# 1. Clone the repo. Replace with your fork URL.
!git clone https://github.com/YOUR-USERNAME/sentinel.git
%cd sentinel

In [ ]:
# 2. Install the ML stack. Colab already has numpy/pandas/sklearn.
!pip install -q xgboost lightgbm shap mapie fairlearn pyarrow

In [ ]:
# 3. Generate the dataset (the NIJ host is currently unreachable, so this synthesises a
#    realistic NIJ-schema dataset). If you want to use a real NIJ CSV instead, drop it at
#    data/raw/nij_recidivism.csv before running this.
!python -m pipelines.data_prep

In [ ]:
# 4. Train. Look for Brier < 0.20, AUC-ROC > 0.75, and SPD/DI within fairness thresholds.
!python -m pipelines.train

In [ ]:
# 5. Sanity-check: load the model and score one row end-to-end.
import pickle, json, pandas as pd
model = pickle.load(open('models/model.pkl', 'rb'))
schema = json.load(open('models/schema.json'))
metrics = json.load(open('models/metrics.json'))
print('test metrics:', {k: v for k, v in metrics.items() if k != 'calibration' and k != 'shap_top_global'})
print()
print('top 5 features by mean |SHAP|:')
for f in metrics['shap_top_global'][:5]:
    print(f"  {f['mean_abs_shap']:.3f}  {f['feature']}")

df = pd.read_parquet('data/processed/offenders.parquet').drop(columns=[schema['target']])
proba = model.predict_proba(df.head(1))[0, 1]
print(f"\nsample row probability of recidivism: {proba:.3f}")

In [ ]:
# 6. Bundle the artifacts so you can download them and commit to the repo.
!cd models && tar czf ../sentinel-models.tar.gz model.pkl mapie.pkl schema.json metrics.json fairness.json
from google.colab import files
files.download('sentinel-models.tar.gz')

## After download

On your machine, from the repo root:

```bash
tar xzf ~/Downloads/sentinel-models.tar.gz -C models/
git add models/*.pkl models/*.json
git commit -m "feat(model): trained ensemble + conformal + fairness audit"
docker compose up -d --build predict
```

The predict service will pick up `models/model.pkl` on startup. The heuristic in `services/predict/app/pipeline.py` becomes the fallback only when no artifact is present.